# Activity Contribution Analysis

This notebook computes the percentage contribution of Combat, Collection, and Exploration activities for each telemetry window based on normalized features.

In [1]:
import pandas as pd
import os
import numpy as np

INPUT_PATH = 'data/processed/3_normalized_telemetry.csv'
OUTPUT_PATH = 'data/processed/4_activity_contributions.csv'

print("Loading normalized telemetry data...")
if os.path.exists(INPUT_PATH):
    df = pd.read_csv(INPUT_PATH)
    print(f"Loaded {len(df)} rows.")
    
    # Define Categories
    # Using both top-level and rawJson keys if present to catch all variants
    F_COMBAT = ['enemiesHit', 'damageDone', 'timeInCombat', 'kills', 
                'rawJson.enemies_hit', 'rawJson.damage_done', 'rawJson.time_in_combat', 'rawJson.kills']
    F_COLLECT = ['itemsCollected', 'pickupAttempts', 'timeNearInteractables',
                 'rawJson.items_collected', 'rawJson.pickup_attempts', 'rawJson.time_near_interactables']
    F_EXPLORE = ['distanceTraveled', 'timeSprinting', 'timeOutOfCombat',
                 'rawJson.distance_traveled', 'rawJson.time_out_of_combat', 'rawJson.time_sprinting']
    
    # Filter to cols actually in DF
    cols_combat = [c for c in F_COMBAT if c in df.columns]
    cols_collect = [c for c in F_COLLECT if c in df.columns]
    cols_explore = [c for c in F_EXPLORE if c in df.columns]
    
    print(f"Combat Features: {cols_combat}")
    print(f"Collection Features: {cols_collect}")
    print(f"Exploration Features: {cols_explore}")
    
    # Compute Raw Scores (Sum of normalized features)
    df['score_combat'] = df[cols_combat].sum(axis=1)
    df['score_collect'] = df[cols_collect].sum(axis=1)
    df['score_explore'] = df[cols_explore].sum(axis=1)
    
    # Compute Total Score
    df['score_total'] = df['score_combat'] + df['score_collect'] + df['score_explore']
    
    # Compute Percentages
    # Avoid Division by Zero
    mask_nonzero = df['score_total'] > 0
    
    df['pct_combat'] = 0.0
    df['pct_collect'] = 0.0
    df['pct_explore'] = 0.0
    
    df.loc[mask_nonzero, 'pct_combat'] = df.loc[mask_nonzero, 'score_combat'] / df.loc[mask_nonzero, 'score_total']
    df.loc[mask_nonzero, 'pct_collect'] = df.loc[mask_nonzero, 'score_collect'] / df.loc[mask_nonzero, 'score_total']
    df.loc[mask_nonzero, 'pct_explore'] = df.loc[mask_nonzero, 'score_explore'] / df.loc[mask_nonzero, 'score_total']
    
    # Verification Stats
    print("\n--- Percentage Stats ---")
    print(df[['pct_combat', 'pct_collect', 'pct_explore']].describe())
    

    # Compute Deltas (Change from previous window)
    # Group by userId to ensure lag is per-player
    print("Computing Deltas...")
    # Ensure Strict Sorting by User and Time before computing lag
    t_col = 'timestamp' if 'timestamp' in df.columns else 'time'
    if t_col in df.columns:
        # Helper for sort
        df['sort_helper'] = pd.to_numeric(df[t_col], errors='coerce')
        if df['sort_helper'].isna().all():
             df['sort_helper'] = pd.to_datetime(df[t_col], errors='coerce')
        df.sort_values(by=['userId', 'sort_helper'], inplace=True)
        df.drop(columns=['sort_helper'], inplace=True)
    # Ensure sort by user+time
    cols_pct = ['pct_combat', 'pct_collect', 'pct_explore']
    
    # Calculate Diff
    # Fillna(0) for the first row of each user
    df[['delta_combat', 'delta_collect', 'delta_explore']] = (
        df.groupby('userId')[cols_pct].diff().fillna(0)
    )
    
    # Verification for Deltas
    print(df[['delta_combat', 'delta_collect', 'delta_explore']].describe())
    
    # Export
    df.to_csv(OUTPUT_PATH, index=False)
    print(f"\nSaved activity contributions to {OUTPUT_PATH}")
else:
    print(f"Error: {INPUT_PATH} not found.")

Loading normalized telemetry data...
Loaded 2715 rows.
Combat Features: ['enemiesHit', 'damageDone', 'timeInCombat', 'kills', 'rawJson.enemies_hit', 'rawJson.damage_done', 'rawJson.time_in_combat', 'rawJson.kills']
Collection Features: ['itemsCollected', 'pickupAttempts', 'timeNearInteractables', 'rawJson.items_collected', 'rawJson.pickup_attempts', 'rawJson.time_near_interactables']
Exploration Features: ['distanceTraveled', 'timeSprinting', 'timeOutOfCombat', 'rawJson.distance_traveled', 'rawJson.time_out_of_combat', 'rawJson.time_sprinting']

--- Percentage Stats ---
        pct_combat  pct_collect  pct_explore
count  2715.000000  2715.000000  2715.000000
mean      0.238980     0.183790     0.577231
std       0.285613     0.164153     0.262641
min       0.000000     0.000000     0.000000
25%       0.000000     0.000000     0.374003
50%       0.074275     0.171730     0.576184
75%       0.464753     0.304840     0.761720
max       1.000000     0.688789     1.000000
Computing Deltas..